# Olist Marketplace – Silver Cleaning
## Customers, Orders & Order Reviews

This notebook combines the three existing preprocessing pipelines into one reproducible Silver-layer workflow.

### Objectives
- Load the three Bronze datasets once.
- Keep Customers, Orders, and Order Reviews clearly separated.
- Preserve the existing transformation and validation logic of the three individual notebooks.
- Use the validated Silver Customers dataset to enrich Orders with `customer_unique_id`.
- Export exactly three Silver CSV files at the end of the notebook.

### Silver outputs
1. `silver_customers.csv`
2. `silver_orders.csv`
3. `silver_order_reviews_en.csv`

No records are intentionally removed by the preprocessing logic. Original source identifiers and relevant source values are preserved.

## 1. Common Setup & Bronze Data Load

Imports, project paths, directory creation, and Bronze data loading are defined only once in this notebook.

In [ ]:
# Common Setup & Bronze Data Load

# import pandas as pd
# import numpy as np
# from pathlib import Path

# pd.set_option("display.max_columns", None)
# pd.set_option("display.max_rows", 100)


# # --------------------------------------------------
# # Project paths
# # --------------------------------------------------

# def find_project_root():
#     """
#     Find the project root by looking for the expected
#     datasets/raw and datasets/processed directories.
#     """
#     current_path = Path.cwd()

#     for path in [current_path, *current_path.parents]:
#         if (
#             (path / "datasets" / "raw").exists()
#             and (path / "datasets" / "processed").exists()
#         ):
#             return path

#     raise FileNotFoundError(
#         "Project root could not be found. "
#         "Expected 'datasets/raw' and 'datasets/processed' directories."
#     )


# PROJECT_ROOT = find_project_root()

# RAW_PATH = PROJECT_ROOT / "datasets" / "raw"
# PROCESSED_PATH = PROJECT_ROOT / "datasets" / "processed"

# PROCESSED_PATH.mkdir(parents=True, exist_ok=True)


# # --------------------------------------------------
# # Load Bronze datasets
# # --------------------------------------------------

# bronze_customers = pd.read_csv(
#     RAW_PATH / "olist_customers_dataset.csv"
# )

# bronze_orders = pd.read_csv(
#     RAW_PATH / "olist_orders_dataset.csv"
# )

# bronze_order_reviews = pd.read_csv(
#     RAW_PATH / "olist_order_reviews_dataset_en.csv"
# )


# # --------------------------------------------------
# # Store original row counts for final validation
# # --------------------------------------------------

# customers_raw_row_count = len(bronze_customers)
# orders_raw_row_count = len(bronze_orders)
# order_reviews_raw_row_count = len(bronze_order_reviews)


# # --------------------------------------------------
# # Initial overview
# # --------------------------------------------------

# print(f"Project root: {PROJECT_ROOT}")
# print(f"Raw path: {RAW_PATH}")
# print(f"Processed path: {PROCESSED_PATH}")

# print("\nBronze dataset shapes:")
# print(f"Customers: {bronze_customers.shape}")
# print(f"Orders: {bronze_orders.shape}")
# print(f"Order reviews: {bronze_order_reviews.shape}")

Project root: c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis
Raw path: c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis\datasets\raw
Processed path: c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis\datasets\processed

Bronze dataset shapes:
Customers: (99441, 5)
Orders: (99441, 8)
Order reviews: (99224, 9)


In [ ]:
# # ============================================================
# # S3 SETUP & BRONZE DATA LOAD – AWS LAMBDA VERSION
# # ============================================================
# #
# # Data flow:
# #
# # S3 Bucket
# #     └── raw/
# #          ├── olist_customers_dataset.csv
# #          ├── olist_orders_dataset.csv
# #          └── olist_order_reviews_dataset_en.csv
# #
# #                 ↓
# #
# #              AWS Lambda
# #
# #                 ↓
# #
# # S3 Bucket
# #     └── processed/
# #          ├── silver_customers.csv
# #          ├── silver_orders.csv
# #          └── silver_order_reviews_en.csv
# #
# # ============================================================


# import boto3
# from io import StringIO


# # ------------------------------------------------------------
# # S3 Configuration
# # ------------------------------------------------------------

# S3_BUCKET = "dsi-final-project-360964564955-eu-central-1-an"

# RAW_PREFIX = "raw/"
# PROCESSED_PREFIX = "processed/"


# # ------------------------------------------------------------
# # S3 Client
# # ------------------------------------------------------------

# s3 = boto3.client("s3")


# # ------------------------------------------------------------
# # Helper function: Load CSV from S3
# # ------------------------------------------------------------

# def load_csv_from_s3(bucket, key):
#     """
#     Load a CSV file from S3 into a pandas DataFrame.
#     """
    
#     response = s3.get_object(
#         Bucket=bucket,
#         Key=key
#     )
    
#     return pd.read_csv(response["Body"])


# # ------------------------------------------------------------
# # Load Bronze datasets from S3
# # ------------------------------------------------------------

# bronze_customers = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_customers_dataset.csv"
# )

# bronze_orders = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_orders_dataset.csv"
# )

# bronze_order_reviews = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_order_reviews_dataset_en.csv"
# )


# # ------------------------------------------------------------
# # Preserve raw row counts for later validation
# # ------------------------------------------------------------

# customers_raw_row_count = len(bronze_customers)
# orders_raw_row_count = len(bronze_orders)
# order_reviews_raw_row_count = len(bronze_order_reviews)


# # ------------------------------------------------------------
# # Initial overview
# # ------------------------------------------------------------

# print(f"Bronze customers: {bronze_customers.shape}")
# print(f"Bronze orders: {bronze_orders.shape}")
# print(f"Bronze order reviews: {bronze_order_reviews.shape}")

# 2. CUSTOMERS – Silver Preprocessing

**Input:** `olist_customers_dataset.csv`  
**Output:** `silver_customers.csv`

This section retains the customer-record grain and applies the transformations from the original Customers preprocessing notebook.

## 2.1 Data Audit

In [3]:
print("Shape:", bronze_customers.shape)
print("\nColumns:")
print(bronze_customers.columns.tolist())

print("\nData types:")
print(bronze_customers.dtypes)

print("\nMissing values:")
display(bronze_customers.isna().sum())

print("\nDuplicate rows:", bronze_customers.duplicated().sum())

print("\nUnique customer_id:", bronze_customers["customer_id"].nunique())
print("Unique customer_unique_id:", bronze_customers["customer_unique_id"].nunique())
print("customer_id is unique:", bronze_customers["customer_id"].is_unique)

Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values:


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Duplicate rows: 0

Unique customer_id: 99441
Unique customer_unique_id: 96096
customer_id is unique: True


In [4]:
customer_counts = (
    bronze_customers
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False)
)

print("Customers with multiple customer records:", (customer_counts > 1).sum())

repeat_customer_records = customer_counts[customer_counts > 1]

print("\nDistribution of customer record counts:")
display(customer_counts.value_counts().sort_index())

print("\nRepeated customer records summary:")
display(repeat_customer_records.describe())

print("\nTotal records belonging to repeated customers:", repeat_customer_records.sum())

Customers with multiple customer records: 2997

Distribution of customer record counts:


1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


Repeated customer records summary:


count    2997.000000
mean        2.116116
std         0.516610
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        17.000000
dtype: float64


Total records belonging to repeated customers: 6342


In [5]:
print("Customer state distribution:")
display(bronze_customers["customer_state"].value_counts())

print("Unique cities:", bronze_customers["customer_city"].nunique())

print("\nZIP code summary:")
display(bronze_customers["customer_zip_code_prefix"].describe())

print("Unique ZIP code prefixes:", bronze_customers["customer_zip_code_prefix"].nunique())
print("Negative ZIP code prefixes:", (bronze_customers["customer_zip_code_prefix"] < 0).sum())
print("Missing ZIP code prefixes:", bronze_customers["customer_zip_code_prefix"].isna().sum())

Customer state distribution:


customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

Unique cities: 4119

ZIP code summary:


count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64

Unique ZIP code prefixes: 14994
Negative ZIP code prefixes: 0
Missing ZIP code prefixes: 0


## 2.2 Cleaning & Standardization

The raw Customers dataset is preserved and transformations are applied to a separate working copy.

The existing preprocessing logic is retained:
- trim whitespace from identifier and location fields
- standardize `customer_state` to lowercase
- standardize `customer_city` using the existing normalization rules
- correct the identified city value
- convert identifier and ZIP fields to explicit string types
- create the technical Silver key `id_customers`

In [6]:
# Create the Silver working copy
silver_customers = bronze_customers.copy()

# Create technical Silver key
silver_customers.insert(
    0,
    "id_customers",
    range(1, len(silver_customers) + 1)
)

# Trim whitespace from string fields
string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]

for column in string_columns:
    silver_customers[column] = silver_customers[column].str.strip()

# Standardize state values
silver_customers["customer_state"] = (
    silver_customers["customer_state"]
    .str.lower()
)

# Standardize city values
silver_customers["customer_city"] = (
    silver_customers["customer_city"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Replace the identified incorrect city value
silver_customers.loc[
    silver_customers["customer_city"] == "quilometro 14 do mutum",
    "customer_city"
] = "mutum"

# Define explicit data types
silver_customers["customer_id"] = (
    silver_customers["customer_id"].astype("string")
)

silver_customers["customer_unique_id"] = (
    silver_customers["customer_unique_id"].astype("string")
)

silver_customers["customer_city"] = (
    silver_customers["customer_city"].astype("string")
)

silver_customers["customer_state"] = (
    silver_customers["customer_state"].astype("string")
)

silver_customers["customer_zip_code_prefix"] = (
    silver_customers["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

## 2.3 Customers – Data Quality Validation

In [7]:
valid_states = {
    "sp", "sc", "mg", "pr", "rj", "rs", "pa", "go", "es", "ba", "ma", "ms", "ce",
    "df", "rn", "pe", "mt", "am", "ap", "al", "ro", "pb", "to", "pi", "ac", "se",
    "rr"
}

invalid_states = (
    set(silver_customers["customer_state"].dropna().unique())
    - valid_states
)

assert silver_customers["customer_id"].notna().all()
assert silver_customers["customer_id"].is_unique

assert silver_customers["customer_unique_id"].notna().all()

assert silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
assert silver_customers["customer_zip_code_prefix"].str.isnumeric().all()

expected_customers_columns = [
    "id_customers",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

assert silver_customers.columns.tolist() == expected_customers_columns
assert len(silver_customers) == customers_raw_row_count

assert silver_customers["id_customers"].notna().all()
assert silver_customers["id_customers"].is_unique
assert silver_customers["id_customers"].min() == 1
assert silver_customers["id_customers"].max() == len(silver_customers)

assert silver_customers["customer_city"].str.match(r"^[a-z\s]+$").all()
assert (
    silver_customers["customer_city"]
    == silver_customers["customer_city"].str.lower()
).all()

assert silver_customers["customer_state"].notna().all()
assert silver_customers.isna().sum().sum() == 0
assert invalid_states == set()

print("Customers Silver validation passed.")
print(f"Rows: {len(silver_customers):,}")
print(f"Columns: {len(silver_customers.columns)}")

Customers Silver validation passed.
Rows: 99,441
Columns: 6


In [8]:
# Additional customer diagnostics retained from the original notebook

character_counts = (
    silver_customers["customer_city"]
    .dropna()
    .astype(str)
    .str.cat(sep="")
)

character_counts = (
    pd.Series(list(character_counts))
    .value_counts()
    .sort_index()
)

for character, count in character_counts.items():
    display_character = "<SPACE>" if character == " " else character
    print(f"{display_character}: {count}")

<SPACE>: 75101
a: 169618
b: 22912
c: 29243
d: 32618
e: 67027
f: 5184
g: 14790
h: 8905
i: 78753
j: 13731
k: 23
l: 44814
m: 19075
n: 45721
o: 126531
p: 37119
q: 2049
r: 76496
s: 62903
t: 33274
u: 44916
v: 9655
w: 39
x: 1133
y: 311
z: 6480


In [9]:
length_analysis = []

for column in silver_customers.columns:
    values = silver_customers[column].dropna().astype(str)
    lengths = values.str.len()

    length_analysis.append({
        "column": column,
        "data_type": str(silver_customers[column].dtype),
        "min_length": lengths.min(),
        "max_length": lengths.max(),
        "shortest_value": values.loc[lengths.idxmin()],
        "longest_value": values.loc[lengths.idxmax()]
    })

length_analysis = pd.DataFrame(length_analysis)
length_analysis

,column,data_type,min_length,max_length,shortest_value,longest_value
0,id_customers,int64,1,5,1,10000
1,customer_id,string,32,32,06b8999e2fba1a1fbc88172c00ba8bc7,06b8999e2fba1a1fbc88172c00ba8bc7
2,customer_unique_id,string,32,32,861eff4711a542e4b93843c6dd7febb0,861eff4711a542e4b93843c6dd7febb0
3,customer_zip_code_prefix,string,5,5,14409,14409
4,customer_city,string,3,32,uba,vila bela da santissima trindade
5,customer_state,string,2,2,sp,sp


# 3. ORDERS – Silver Preprocessing

**Input:** `olist_orders_dataset.csv`  
**Output:** `silver_orders.csv`

The Orders section uses the already validated in-memory `silver_customers` dataset. No second Customers file load is required.

## 3.1 Data Audit & Key Validation

In [10]:
print("Shape:", bronze_orders.shape)
print("\nColumns:")
print(bronze_orders.columns.tolist())

print("\nData types:")
print(bronze_orders.dtypes)

print("\nMissing values:")
display(
    bronze_orders.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
    .assign(
        missing_pct=lambda x: (
            x["missing_count"] / len(bronze_orders) * 100
        ).round(2)
    )
)

print("\nDuplicate rows:", bronze_orders.duplicated().sum())

print("\nUnique order_ids:", bronze_orders["order_id"].nunique())
print("Total rows:", len(bronze_orders))
print("Duplicate order_ids:", bronze_orders["order_id"].duplicated().sum())

customer_id_match = bronze_orders["customer_id"].isin(
    bronze_customers["customer_id"]
)

print("Orders with valid customer_id:", customer_id_match.sum())
print("Orders with missing customer reference:", (~customer_id_match).sum())

print("\nOrder status distribution:")
display(
    bronze_orders["order_status"]
    .value_counts()
    .to_frame("order_count")
    .assign(
        percentage=lambda x: (
            x["order_count"] / len(bronze_orders) * 100
        ).round(2)
    )
)

Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values:


,missing_count,missing_pct
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_id,0,0.00
order_purchase_timestamp,0,0.00
order_status,0,0.00
customer_id,0,0.00
order_estimated_delivery_date,0,0.00



Duplicate rows: 0

Unique order_ids: 99441
Total rows: 99441
Duplicate order_ids: 0
Orders with valid customer_id: 99441
Orders with missing customer reference: 0

Order status distribution:


,order_count,percentage
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


## 3.2 Temporal Validation & Quality Flags

In [11]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

bronze_orders[date_columns] = bronze_orders[date_columns].apply(
    pd.to_datetime
)

print(
    "Approved before purchase:",
    (
        bronze_orders["order_approved_at"]
        < bronze_orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Carrier before purchase:",
    (
        bronze_orders["order_delivered_carrier_date"]
        < bronze_orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before purchase:",
    (
        bronze_orders["order_delivered_customer_date"]
        < bronze_orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before carrier:",
    (
        bronze_orders["order_delivered_customer_date"]
        < bronze_orders["order_delivered_carrier_date"]
    ).sum()
)

Approved before purchase: 0
Carrier before purchase: 166
Customer delivery before purchase: 0
Customer delivery before carrier: 23


In [12]:
missing_pct_by_status = (
    bronze_orders
    .groupby("order_status")[date_columns]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

display(missing_pct_by_status)

delivered_missing_dates = bronze_orders[
    (bronze_orders["order_status"] == "delivered")
    & bronze_orders[date_columns].isna().any(axis=1)
].copy()

print(
    "Delivered orders with missing timestamps:",
    len(delivered_missing_dates)
)

display(
    delivered_missing_dates[
        ["order_id", "order_status"] + date_columns
    ].head(5)
)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


Delivered orders with missing timestamps: 23


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16


In [13]:
# Create temporal quality flags
bronze_orders["carrier_before_purchase_flag"] = (
    bronze_orders["order_delivered_carrier_date"]
    < bronze_orders["order_purchase_timestamp"]
)

bronze_orders["delivery_before_carrier_flag"] = (
    bronze_orders["order_delivered_customer_date"]
    < bronze_orders["order_delivered_carrier_date"]
)

# Create missing timestamp flags
bronze_orders["missing_approval_flag"] = (
    bronze_orders["order_approved_at"].isna()
)

bronze_orders["missing_carrier_date_flag"] = (
    bronze_orders["order_delivered_carrier_date"].isna()
)

bronze_orders["missing_delivery_date_flag"] = (
    bronze_orders["order_delivered_customer_date"].isna()
)

quality_flags = [
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

quality_flag_summary = pd.DataFrame({
    "flag": quality_flags,
    "count": [
        bronze_orders[column].sum()
        for column in quality_flags
    ]
})

quality_flag_summary["percentage"] = (
    quality_flag_summary["count"]
    / len(bronze_orders)
    * 100
).round(3)

display(quality_flag_summary)

,flag,count,percentage
0,carrier_before_purchase_flag,166,0.167
1,delivery_before_carrier_flag,23,0.023
2,missing_approval_flag,160,0.161
3,missing_carrier_date_flag,1783,1.793
4,missing_delivery_date_flag,2965,2.982


## 3.3 Create Silver Orders Dataset

In [ ]:
silver_orders = bronze_orders.copy()

silver_orders.insert(
    0,
    "id_orders",
    range(1, len(silver_orders) + 1)
)

expected_orders_columns = [
    "id_orders",
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

silver_orders = silver_orders[expected_orders_columns]

print(f"Rows: {len(silver_orders):,}")
print(f"Columns: {len(silver_orders.columns)}")
print(f"Unique id_orders: {silver_orders['id_orders'].nunique():,}")

Rows: 99,441
Columns: 15
Unique id_orders: 99,441


## 3.5 Orders – Silver Data Quality Validation

In [ ]:
assert silver_orders.columns.tolist() == expected_orders_columns
assert len(silver_orders) == orders_raw_row_count

assert silver_orders["id_orders"].notna().all()
assert silver_orders["id_orders"].is_unique
assert silver_orders["id_orders"].min() == 1
assert silver_orders["id_orders"].max() == len(silver_orders)

assert silver_orders["order_id"].notna().all()
assert silver_orders["order_id"].is_unique
assert silver_orders["customer_id"].notna().all()
assert silver_orders["order_status"].notna().all()
assert silver_orders["order_purchase_timestamp"].notna().all()

assert all(
    pd.api.types.is_datetime64_any_dtype(silver_orders[column])
    for column in date_columns
)

assert all(
    silver_orders[column].dtype == bool
    for column in quality_flags
)

assert (
    silver_orders["carrier_before_purchase_flag"]
    == (
        silver_orders["order_delivered_carrier_date"]
        < silver_orders["order_purchase_timestamp"]
    )
).all()

assert (
    silver_orders["delivery_before_carrier_flag"]
    == (
        silver_orders["order_delivered_customer_date"]
        < silver_orders["order_delivered_carrier_date"]
    )
).all()

assert (
    silver_orders["missing_approval_flag"]
    == silver_orders["order_approved_at"].isna()
).all()

assert (
    silver_orders["missing_carrier_date_flag"]
    == silver_orders["order_delivered_carrier_date"].isna()
).all()

assert (
    silver_orders["missing_delivery_date_flag"]
    == silver_orders["order_delivered_customer_date"].isna()
).all()

silver_quality_summary = (
    silver_orders[quality_flags]
    .sum()
    .to_frame("flagged_count")
)

silver_quality_summary["flagged_pct"] = (
    silver_quality_summary["flagged_count"]
    / len(silver_orders)
    * 100
).round(3)

print("Orders Silver validation passed.")
print(f"Rows: {len(silver_orders):,}")
print(f"Unique order_ids: {silver_orders['order_id'].nunique():,}")

Orders Silver validation passed.
Rows: 99,441
Unique order_ids: 99,441


In [17]:
character_counts = (
    silver_orders["order_status"]
    .dropna()
    .astype(str)
    .str.cat(sep="")
)

character_counts = (
    pd.Series(list(character_counts))
    .value_counts()
    .sort_index()
)

for character, count in character_counts.items():
    display_character = "<SPACE>" if character == " " else character
    print(f"{display_character}: {count}")

order_status_counts = (
    silver_orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="count")
)

display(order_status_counts)

a: 2459
b: 609
c: 1870
d: 195009
e: 293027
g: 301
h: 1107
i: 99123
l: 98321
n: 1849
o: 617
p: 2519
r: 96786
s: 1709
t: 5
u: 609
v: 97403


,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


# 4. ORDER REVIEWS – Silver Preprocessing

**Input:** `olist_order_reviews_dataset_en.csv`  
**Output:** `silver_order_reviews_en.csv`

This section preserves the original review-level grain and applies the exact text normalization, translation completion, and datetime transformations from the existing Order Reviews preprocessing notebook.

## 4.1 Data Understanding & Quality Assessment

In [18]:
print("Shape:", bronze_order_reviews.shape)
print("\nColumns:")
print(bronze_order_reviews.columns.tolist())

print("\nData types:")
print(bronze_order_reviews.dtypes)

print("\nMissing values:")
display(
    bronze_order_reviews.isna()
    .sum()
    .to_frame("missing_count")
    .assign(
        missing_pct=lambda x: (
            x["missing_count"] / len(bronze_order_reviews) * 100
        )
    )
)

print("\nDuplicate rows:", bronze_order_reviews.duplicated().sum())
print("Unique review_id:", bronze_order_reviews["review_id"].nunique())
print("Rows with duplicated review_id:", bronze_order_reviews["review_id"].duplicated().sum())
print("Unique order_id:", bronze_order_reviews["order_id"].nunique())
print("Rows with duplicated order_id:", bronze_order_reviews["order_id"].duplicated().sum())

print("\nReview score distribution:")
display(bronze_order_reviews["review_score"].value_counts().sort_index())

Shape: (99224, 9)

Columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_title_en', 'review_comment_message', 'review_comment_message_en', 'review_creation_date', 'review_answer_timestamp']

Data types:
review_id                      str
order_id                       str
review_score                 int64
review_comment_title           str
review_comment_title_en        str
review_comment_message         str
review_comment_message_en      str
review_creation_date           str
review_answer_timestamp        str
dtype: object

Missing values:


,missing_count,missing_pct
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_title_en,87660,88.345562
review_comment_message,58247,58.702532
review_comment_message_en,58277,58.732766
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000



Duplicate rows: 0
Unique review_id: 98410
Rows with duplicated review_id: 814
Unique order_id: 98673
Rows with duplicated order_id: 551

Review score distribution:


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [19]:
expected_review_input_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert bronze_order_reviews.columns.tolist() == expected_review_input_columns

mandatory_review_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

assert bronze_order_reviews[mandatory_review_columns].notna().all().all()
assert bronze_order_reviews["review_score"].between(1, 5).all()

review_creation_dt = pd.to_datetime(
    bronze_order_reviews["review_creation_date"],
    errors="coerce"
)

review_answer_dt = pd.to_datetime(
    bronze_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

assert review_creation_dt.notna().all()
assert review_answer_dt.notna().all()

answer_before_creation = review_answer_dt < review_creation_dt
assert not answer_before_creation.any()

print("Order Reviews Bronze validation passed.")

Order Reviews Bronze validation passed.


## 4.2 Create Silver Dataset & Normalize Text

In [20]:
silver_order_reviews = bronze_order_reviews.copy()

silver_order_reviews["id_order_reviews"] = range(
    1,
    len(silver_order_reviews) + 1
)

silver_order_reviews.insert(
    0,
    "id_order_reviews",
    silver_order_reviews.pop("id_order_reviews")
)

assert silver_order_reviews["id_order_reviews"].is_unique
assert silver_order_reviews["id_order_reviews"].min() == 1
assert silver_order_reviews["id_order_reviews"].max() == len(silver_order_reviews)

comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

for column in comment_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

translation_columns = [
    "review_comment_title_en",
    "review_comment_message_en"
]

for column in translation_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [21]:
# Complete the defined English translation mappings
silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota dez",
    "review_comment_title_en"
] = "ranking ten"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota",
    "review_comment_title_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nota",
    "review_comment_message_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nenhuma",
    "review_comment_title_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nenhuma",
    "review_comment_message_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nd",
    "review_comment_message_en"
] = "not available"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "no h",
    "review_comment_message_en"
] = "there is no"

# Validate normalized text fields
for column in comment_columns + translation_columns:
    non_null_values = silver_order_reviews[column].dropna()

    assert (
        non_null_values
        == non_null_values.str.strip()
    ).all()

    assert (
        non_null_values
        == non_null_values.str.lower()
    ).all()

    assert (
        non_null_values
        .str.match(r"^[a-z\s.,!?]+$")
        .all()
    )

assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

## 4.3 Datetime Transformation & Validation

In [22]:
datetime_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in datetime_columns:
    silver_order_reviews[column] = pd.to_datetime(
        silver_order_reviews[column],
        errors="raise"
    )

assert silver_order_reviews[datetime_columns].notna().all().all()

answer_before_creation = (
    silver_order_reviews["review_answer_timestamp"]
    < silver_order_reviews["review_creation_date"]
)

assert not answer_before_creation.any()

assert len(silver_order_reviews) == len(bronze_order_reviews)
assert (
    silver_order_reviews["review_id"].nunique()
    == bronze_order_reviews["review_id"].nunique()
)
assert (
    silver_order_reviews["order_id"].nunique()
    == bronze_order_reviews["order_id"].nunique()
)

print("Order Reviews transformation validation passed.")

Order Reviews transformation validation passed.


## 4.4 Order Reviews – Silver Data Quality Validation

In [23]:
expected_order_reviews_columns = [
    "id_order_reviews",
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert silver_order_reviews.columns.tolist() == expected_order_reviews_columns
assert len(silver_order_reviews) == order_reviews_raw_row_count

assert silver_order_reviews["id_order_reviews"].notna().all()
assert silver_order_reviews["id_order_reviews"].is_unique
assert silver_order_reviews["id_order_reviews"].min() == 1
assert silver_order_reviews["id_order_reviews"].max() == len(silver_order_reviews)

assert silver_order_reviews[mandatory_review_columns].notna().all().all()
assert silver_order_reviews["review_score"].between(1, 5).all()

assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_creation_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_answer_timestamp"]
)

assert (
    silver_order_reviews["review_answer_timestamp"]
    >= silver_order_reviews["review_creation_date"]
).all()

print("Order Reviews Silver validation passed.")
print(f"Rows: {len(silver_order_reviews):,}")
print(f"Columns: {len(silver_order_reviews.columns)}")

Order Reviews Silver validation passed.
Rows: 99,224
Columns: 10


In [24]:
columns_to_check = [
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_score"
]

for column in columns_to_check:
    print(f"\n--- {column} ---")

    character_counts = (
        silver_order_reviews[column]
        .dropna()
        .astype(str)
        .str.cat(sep="")
    )

    character_counts = (
        pd.Series(list(character_counts))
        .value_counts()
        .sort_index()
    )

    for character, count in character_counts.items():
        display_character = "<SPACE>" if character == " " else character
        print(f"{display_character}: {count}")


--- review_comment_title ---
<SPACE>: 9794
!: 1029
,: 131
.: 692
?: 46
a: 8062
b: 2364
c: 5415
d: 7138
e: 16791
f: 1202
g: 1451
h: 439
i: 6962
j: 193
k: 193
l: 2460
m: 8228
n: 7043
o: 18733
p: 4360
q: 294
r: 9657
s: 3627
t: 8316
u: 4872
v: 704
w: 33
x: 825
y: 15
z: 313

--- review_comment_title_en ---
<SPACE>: 12088
!: 1035
,: 254
.: 685
?: 48
a: 5715
b: 875
c: 6737
d: 9700
e: 19614
f: 1613
g: 5014
h: 3229
i: 8720
j: 29
k: 640
l: 5690
m: 5897
n: 6588
o: 10980
p: 2819
q: 164
r: 11171
s: 3389
t: 9295
u: 2706
v: 2991
w: 834
x: 877
y: 3125
z: 49

--- review_comment_message ---
<SPACE>: 427808
!: 10550
,: 22485
.: 40521
?: 1161
a: 236423
b: 29835
c: 81000
d: 124507
e: 284716
f: 26224
g: 35300
h: 19748
i: 134458
j: 6899
k: 1970
l: 44548
m: 110036
n: 117117
o: 296574
p: 87373
q: 18007
r: 168793
s: 109594
t: 136559
u: 99477
v: 23977
w: 282
x: 5882
y: 497
z: 13302

--- review_comment_message_en ---
<SPACE>: 497045
!: 10522
,: 30305
.: 53900
?: 1242
a: 165991
b: 34279
c: 89482
d: 151027
e: 3447

# 5. Final Validation & Common Silver Export

All three Silver datasets are validated before export.

The export is intentionally located in one single block at the end of the notebook so that executing the notebook produces exactly these three files:
- `silver_customers.csv`
- `silver_orders.csv`
- `silver_order_reviews_en.csv`

In [ ]:
# Final cross-dataset validation

assert len(silver_customers) == customers_raw_row_count
assert len(silver_orders) == orders_raw_row_count
assert len(silver_order_reviews) == order_reviews_raw_row_count

assert silver_customers["customer_id"].is_unique
assert silver_orders["order_id"].is_unique
assert silver_order_reviews["id_order_reviews"].is_unique

print("All three Silver datasets passed the final cross-dataset validation.")

All three Silver datasets passed the final cross-dataset validation.


In [ ]:
# # Common Silver export
# # This is the only export block in the notebook.

# customers_output = PROCESSED_PATH / "silver_customers.csv"
# orders_output = PROCESSED_PATH / "silver_orders.csv"
# order_reviews_output = PROCESSED_PATH / "silver_order_reviews_en.csv"

# silver_customers.to_csv(
#     customers_output,
#     index=False
# )

# silver_orders.to_csv(
#     orders_output,
#     index=False
# )

# silver_order_reviews.to_csv(
#     order_reviews_output,
#     index=False
# )

# print("Silver datasets exported successfully:")
# print(f"- Customers:     {customers_output}")
# print(f"- Orders:        {orders_output}")
# print(f"- Order Reviews: {order_reviews_output}")

Silver datasets exported successfully:
- Customers:     c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis\datasets\processed\silver_customers.csv
- Orders:        c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis\datasets\processed\silver_orders.csv
- Order Reviews: c:\Users\bigd\Desktop\DSI - Weiterbildung\olist_marketplace_analysis\olist-marketplace-analysis\datasets\processed\silver_order_reviews_en.csv


In [ ]:
# # ============================================================
# # S3 EXPORT – AWS LAMBDA VERSION
# # ============================================================
# #
# # Data flow:
# #
# # Silver DataFrames
# #        ↓
# #    AWS Lambda
# #        ↓
# # S3 Bucket
# #     └── processed/
# #          ├── silver_customers.csv
# #          ├── silver_orders.csv
# #          └── silver_order_reviews_en.csv
# #
# # ============================================================


# import boto3
# from io import StringIO


# # ------------------------------------------------------------
# # S3 Configuration
# # ------------------------------------------------------------

# S3_BUCKET = "dsi-final-project-360964564955-eu-central-1-an"

# PROCESSED_PREFIX = "processed/"


# # ------------------------------------------------------------
# # S3 Client
# # ------------------------------------------------------------

# s3 = boto3.client("s3")


# # ------------------------------------------------------------
# # Helper function: Upload DataFrame to S3 as CSV
# # ------------------------------------------------------------

# def save_csv_to_s3(df, bucket, key):
#     """
#     Upload a pandas DataFrame to S3 as a CSV file.
#     """

#     csv_buffer = StringIO()

#     df.to_csv(
#         csv_buffer,
#         index=False
#     )

#     s3.put_object(
#         Bucket=bucket,
#         Key=key,
#         Body=csv_buffer.getvalue(),
#         ContentType="text/csv"
#     )

#     print(f"Successfully uploaded: s3://{bucket}/{key}")


# # ------------------------------------------------------------
# # Export Silver datasets to S3
# # ------------------------------------------------------------

# save_csv_to_s3(
#     silver_customers,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_customers.csv"
# )

# save_csv_to_s3(
#     silver_orders,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_orders.csv"
# )

# save_csv_to_s3(
#     silver_order_reviews,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_order_reviews_en.csv"
# )


# # ------------------------------------------------------------
# # Export confirmation
# # ------------------------------------------------------------

# print("All Silver datasets successfully exported to S3.")

## 6. Final Output Summary

The notebook produces three validated Silver datasets from the three Bronze inputs.

### Customers
Preserves the customer-record grain and creates `id_customers`.

### Orders
Preserves the order-level grain, creates `id_orders`, adds `customer_unique_id` through the validated Customers mapping, and retains explicit temporal and missing-timestamp quality flags.

### Order Reviews
Preserves the review-level grain, creates `id_order_reviews`, normalizes Portuguese and English text fields, applies the defined English translation mappings, and converts review timestamps to datetime.

Business KPIs, analytical features, and Gold-layer transformations are intentionally excluded from this Silver-layer notebook.